# Temporal LLM Integration Tests

> Integration of LLM reasoning with temporal knowledge graphs

In [ ]:
#| default_exp tools.temporal
#| hide
from fastcore.test import test_ne, test_eq
from cogitarelink.core.debug import get_logger
from cogitarelink.reason.afford import AffordanceScanner
from cogitarelink.reason.sandbox import reason_over
from cogitarelink.tools.reason import reason_tool, FUNCTION_SPEC
from datetime import datetime, timezone, timedelta
import json

In [ ]:
#| export
from typing import Union, List, Dict, Tuple, Optional
from pydantic import BaseModel, ConfigDict
from rdflib import Graph, Namespace, URIRef, Literal as RDFLiteral, BNode
from rdflib.namespace import RDF, RDFS, OWL, XSD, TIME
from pyshacl import validate
import uuid
from cogitarelink.core.graph import GraphManager
from cogitarelink.reason.prov import wrap_patch_with_prov
from cogitarelink.core.temporal import (
    TimeInstant, TimeInterval, Event, InstantReification, 
    IntervalReification, LifespanReification, Namespaces,
    infer_temporal_relations, event_to_jsonld, create_test_events
)

In [ ]:
#| export
log = get_logger("temporal.llm")

In [ ]:
#| export
TEMPORAL_FUNCTION_SPEC: Dict = {
  "name": "temporal_reasoning",
  "description": "Reason over temporal events and their relationships",
  "parameters": {
    "type": "object",
    "properties": {
      "operation": {
        "type": "string",
        "enum": ["analyze_relations", "calculate_duration", "check_overlap", "find_participants"],
        "description": "Operation to perform on temporal data"
      },
      "event_data": {
        "type": "string",
        "description": "JSON-LD event data to analyze"
      },
      "query_params": {
        "type": "object",
        "description": "Additional parameters for the operation"
      }
    },
    "required": ["operation", "event_data"]
  }
}

In [ ]:
#| export
def temporal_reasoning_tool(operation: str, event_data: str, query_params: Optional[Dict] = None) -> str:
    """Tool for temporal reasoning over event data
    
    Args:
        operation: Operation to perform (analyze_relations, calculate_duration, check_overlap, find_participants)
        event_data: JSON-LD event data to analyze
        query_params: Additional parameters for the operation
        
    Returns:
        Results of the temporal reasoning operation as JSON string
    """
    # Parse the event data from JSON-LD
    try:
        event_json = json.loads(event_data) if isinstance(event_data, str) else event_data
    except json.JSONDecodeError:
        return json.dumps({"error": "Invalid JSON-LD data"})
    
    # Create a graph from the JSON-LD
    g = Graph()
    try:
        g.parse(data=json.dumps(event_json), format="json-ld")
    except Exception as e:
        return json.dumps({"error": f"Error parsing JSON-LD: {str(e)}"})
    
    # Apply temporal inference
    inferred_graph = infer_temporal_relations(g)
    
    # Process according to operation
    if operation == "analyze_relations":
        return _analyze_temporal_relations(inferred_graph, query_params or {})
    elif operation == "calculate_duration":
        return _calculate_durations(inferred_graph, query_params or {})
    elif operation == "check_overlap":
        return _check_event_overlap(inferred_graph, query_params or {})
    elif operation == "find_participants":
        return _find_event_participants(inferred_graph, query_params or {})
    else:
        return json.dumps({"error": f"Unknown operation: {operation}"})

In [ ]:
#| export
def _analyze_temporal_relations(graph, params):
    """Analyze temporal relationships between events in the graph"""
    # Query for temporal relationships
    query = """
    PREFIX temp: <https://example.org/temporal-relations#>
    PREFIX event: <https://example.org/event-ontology#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    
    SELECT ?event1 ?event1Label ?relation ?event2 ?event2Label
    WHERE {
        ?event1 a event:Event ;
               rdfs:label ?event1Label ;
               event:hasTimeInterval ?interval1 .
        ?interval1 ?relation ?interval2 .
        ?event2 a event:Event ;
               rdfs:label ?event2Label ;
               event:hasTimeInterval ?interval2 .
        FILTER(STRSTARTS(STR(?relation), STR(temp:)))
        FILTER(?event1 != ?event2)
    }
    ORDER BY ?event1 ?relation ?event2
    """
    
    results = graph.query(query)
    
    relations = []
    for row in results:
        relation_name = str(row.relation).split('#')[-1]
        relations.append({
            "event1": str(row.event1),
            "event1Name": str(row.event1Label),
            "relation": relation_name,
            "event2": str(row.event2),
            "event2Name": str(row.event2Label)
        })
    
    return json.dumps({
        "operation": "analyze_relations",
        "relations": relations,
        "count": len(relations)
    })

In [ ]:
#| export
def _calculate_durations(graph, params):
    """Calculate durations of events in the graph"""
    # Query for event durations
    query = """
    PREFIX event: <https://example.org/event-ontology#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX time: <http://www.w3.org/2006/time#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    
    SELECT ?event ?eventLabel ?start ?end
    WHERE {
        ?event a event:Event ;
               rdfs:label ?eventLabel ;
               event:hasTimeInterval ?interval .
        ?interval time:hasBeginning/time:inXSDDateTime ?start .
        OPTIONAL { ?interval time:hasEnd/time:inXSDDateTime ?end . }
    }
    ORDER BY ?start
    """
    
    results = graph.query(query)
    
    durations = []
    for row in results:
        event_id = str(row.event)
        event_name = str(row.eventLabel)
        start_time = row.start.toPython() if hasattr(row.start, 'toPython') else str(row.start)
        
        if hasattr(row, 'end') and row.end:
            end_time = row.end.toPython() if hasattr(row.end, 'toPython') else str(row.end)
            
            # Calculate duration if both start and end are available
            try:
                start_dt = datetime.fromisoformat(start_time.replace('Z', '+00:00'))
                end_dt = datetime.fromisoformat(end_time.replace('Z', '+00:00'))
                duration_seconds = (end_dt - start_dt).total_seconds()
                duration_str = str(timedelta(seconds=duration_seconds))
            except Exception as e:
                duration_seconds = None
                duration_str = f"Error: {str(e)}"
        else:
            end_time = None
            duration_seconds = None
            duration_str = "Unknown (no end time)"
        
        durations.append({
            "eventId": event_id,
            "eventName": event_name,
            "startTime": start_time,
            "endTime": end_time,
            "durationSeconds": duration_seconds,
            "durationFormatted": duration_str
        })
    
    return json.dumps({
        "operation": "calculate_duration",
        "durations": durations,
        "count": len(durations)
    })

In [ ]:
#| export
def _check_event_overlap(graph, params):
    """Check for overlapping events in the graph"""
    # Query for overlapping events
    query = """
    PREFIX temp: <https://example.org/temporal-relations#>
    PREFIX event: <https://example.org/event-ontology#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    
    SELECT ?event1 ?event1Label ?event2 ?event2Label
    WHERE {
      ?event1 a event:Event ;
             rdfs:label ?event1Label ;
             event:hasTimeInterval ?interval1 .
      ?event2 a event:Event ;
             rdfs:label ?event2Label ;
             event:hasTimeInterval ?interval2 .
      # Get events that overlap in the temporal sense
      {
        ?interval1 temp:overlaps ?interval2 .
      } UNION {
        ?interval1 temp:contains ?interval2 .
      } UNION {
        ?interval1 temp:during ?interval2 .
      }
      FILTER(?event1 != ?event2)
    }
    """
    
    results = graph.query(query)
    
    overlaps = []
    for row in results:
        overlaps.append({
            "event1": str(row.event1),
            "event1Name": str(row.event1Label),
            "event2": str(row.event2),
            "event2Name": str(row.event2Label),
        })
    
    return json.dumps({
        "operation": "check_overlap",
        "overlaps": overlaps,
        "hasOverlaps": len(overlaps) > 0,
        "count": len(overlaps)
    })

In [ ]:
#| export
def _find_event_participants(graph, params):
    """Find participants in events from the graph"""
    # Query for event participants
    query = """
    PREFIX event: <https://example.org/event-ontology#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    
    SELECT ?event ?eventLabel ?participant ?participantName ?role
    WHERE {
        ?event a event:Event ;
               rdfs:label ?eventLabel ;
               event:hasParticipant ?participant .
        OPTIONAL { ?participant rdfs:label ?participantName . }
        OPTIONAL { ?participant event:hasRole ?role . }
    }
    ORDER BY ?event ?participant
    """
    
    results = graph.query(query)
    
    # Organize participants by event
    events = {}
    for row in results:
        event_id = str(row.event)
        if event_id not in events:
            events[event_id] = {
                "eventId": event_id,
                "eventName": str(row.eventLabel),
                "participants": []
            }
        
        participant = {
            "participantId": str(row.participant),
            "participantName": str(row.participantName) if hasattr(row, 'participantName') and row.participantName else None,
            "role": str(row.role) if hasattr(row, 'role') and row.role else None
        }
        
        events[event_id]["participants"].append(participant)
    
    return json.dumps({
        "operation": "find_participants",
        "events": list(events.values()),
        "count": len(events)
    })

In [ ]:
#| export
def integrate_temporal_with_shacl_reasoning(event_data, shapes_turtle=None, query=None):
    """Integrate temporal reasoning with SHACL validation
    
    Args:
        event_data: Event data as JSON-LD
        shapes_turtle: Optional SHACL shapes in turtle format
        query: Optional SPARQL CONSTRUCT query
        
    Returns:
        Combined reasoning results
    """
    # First use the reason_over function from sandbox module
    jsonld_str = json.dumps(event_data) if isinstance(event_data, dict) else event_data
    patch_jsonld, summary = reason_over(
        jsonld=jsonld_str,
        shapes_turtle=shapes_turtle,
        query=query
    )
    
    # Now perform temporal reasoning on the result
    temporal_result = temporal_reasoning_tool(
        operation="analyze_relations",
        event_data=patch_jsonld
    )
    
    return {
        "shacl_summary": summary,
        "temporal_analysis": json.loads(temporal_result),
        "combined_jsonld": json.loads(patch_jsonld)
    }

In [ ]:
# Test case 1: Basic temporal reasoning tool
events = create_test_events()
event_jsonld = event_to_jsonld(events[0])

# Test analyzing temporal relations
analysis_result = temporal_reasoning_tool(
    operation="analyze_relations",
    event_data=json.dumps(event_jsonld)
)

# Parse the result and verify
result_obj = json.loads(analysis_result)
print(f"Analysis result: {json.dumps(result_obj, indent=2)}")

# Verify structure
test_eq("operation" in result_obj, True)
test_eq("relations" in result_obj, True)

In [ ]:
# Test case 2: Calculate durations
duration_result = temporal_reasoning_tool(
    operation="calculate_duration",
    event_data=json.dumps(event_jsonld)
)

# Parse the result and verify
duration_obj = json.loads(duration_result)
print(f"Duration result: {json.dumps(duration_obj, indent=2)}")

# Verify structure
test_eq("operation" in duration_obj, True)
test_eq("durations" in duration_obj, True)
test_eq(len(duration_obj["durations"]) > 0, True)

In [ ]:
# Test case 3: Check for overlapping events
# Create an event that overlaps with the morning meeting
overlapping_event = Event(
    id="event:overlapping1",
    name="Overlapping Meeting",
    description="A meeting that overlaps with the morning meeting",
    startTime=TimeInstant(dateTime=datetime(2025, 6, 15, 9, 30, tzinfo=timezone.utc)),
    endTime=TimeInstant(dateTime=datetime(2025, 6, 15, 10, 30, tzinfo=timezone.utc)),
    location="event:room101"
)

# Convert both events to JSON-LD and combine them
combined_jsonld = {
    "@context": event_jsonld["@context"],
    "@graph": [
        {k: v for k, v in event_jsonld.items() if k != "@context"},
        {k: v for k, v in event_to_jsonld(overlapping_event).items() if k != "@context"}
    ]
}

overlap_result = temporal_reasoning_tool(
    operation="check_overlap",
    event_data=json.dumps(combined_jsonld)
)

# Parse the result and verify
overlap_obj = json.loads(overlap_result)
print(f"Overlap result: {json.dumps(overlap_obj, indent=2)}")

# Verify overlaps were found
test_eq(overlap_obj["hasOverlaps"], True)
test_eq(len(overlap_obj["overlaps"]) > 0, True)

In [ ]:
# Test case 4: Find participants
participants_result = temporal_reasoning_tool(
    operation="find_participants",
    event_data=json.dumps(event_jsonld)
)

# Parse the result and verify
participants_obj = json.loads(participants_result)
print(f"Participants result: {json.dumps(participants_obj, indent=2)}")

# Verify structure
test_eq("events" in participants_obj, True)
test_eq(len(participants_obj["events"]) > 0, True)
test_eq("participants" in participants_obj["events"][0], True)

In [ ]:
# Test case 5: Integration with SHACL reasoning
# Create SHACL shapes for events
event_shapes = """
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix event: <https://example.org/event-ontology#> .

event:EventShape a sh:NodeShape ;
    sh:targetClass event:Event ;
    sh:property [
        sh:path rdfs:label ;
        sh:minCount 1 ;
        sh:datatype xsd:string ;
    ] ;
    sh:property [
        sh:path event:hasTimeInterval ;
        sh:minCount 1 ;
    ] ;
    sh:property [
        sh:path event:hasParticipant ;
        sh:minCount 1 ;
    ] .
"""

# Test integration
integrated_result = integrate_temporal_with_shacl_reasoning(
    event_data=event_jsonld,
    shapes_turtle=event_shapes
)

print(f"SHACL summary: {integrated_result['shacl_summary']}")
print(f"Temporal analysis: {json.dumps(integrated_result['temporal_analysis'], indent=2)}")

# Verify integration structure
test_eq("shacl_summary" in integrated_result, True)
test_eq("temporal_analysis" in integrated_result, True)
test_eq("combined_jsonld" in integrated_result, True)

In [ ]:
# Test case 6: Function specification for LLM tool usage
print("Temporal Function Spec for LLM Tools:")
print(json.dumps(TEMPORAL_FUNCTION_SPEC, indent=2))

# Verify function spec structure
test_eq("name" in TEMPORAL_FUNCTION_SPEC, True)
test_eq("parameters" in TEMPORAL_FUNCTION_SPEC, True)
test_eq("properties" in TEMPORAL_FUNCTION_SPEC["parameters"], True)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()